# 方法2：双模型 Block/Site 级 Causal Patching（Case Study + Aggregate）

本 notebook 目标：
- 同时加载 `standard` 与 `attnres` 两个 checkpoint
- 在同一输入、同一 patch site 条件下做 block/site 级 causal patching
- 先给出单样本 case study，再给出小规模多样本 aggregate 统计

说明：本 notebook 的核心 metric 是 **next-token target logprob**，
patch effect 定义为：
`patched next-token target logprob - corrupted next-token target logprob`。


In [ ]:
import json
import random
import sys
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# ---- Reproducibility ----
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Locate repo root ----
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / "toygpt2").exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError("未找到仓库根目录（应包含 toygpt2 目录）")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_NAME = "method2_block_site_patching"
OUTPUT_DIR = REPO_ROOT / "output" / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def output_path(filename: str) -> Path:
    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _to_serializable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_serializable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if torch.is_tensor(value):
        return value.detach().cpu().tolist()
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return value


def save_json_artifact(payload, filename: str) -> Path:
    path = output_path(filename)
    path.write_text(json.dumps(_to_serializable(payload), ensure_ascii=False, indent=2), encoding='utf-8')
    print('[saved]', path)
    return path


def save_dataframe_artifact(df, filename: str) -> Path:
    path = output_path(filename)
    df.to_csv(path, index=False)
    print('[saved]', path)
    return path


def save_figure_artifact(filename: str, fig=None, *, dpi: int = 180) -> Path:
    fig = plt.gcf() if fig is None else fig
    path = output_path(filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('[saved]', path)
    return path

print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Seed:", SEED)


## 加载双模型 Checkpoint

默认路径：
- `REPO_ROOT/toygpt2_runs/tinystories_dual/standard/ckpt_standard_last.pt`
- `REPO_ROOT/toygpt2_runs/tinystories_dual/attnres/ckpt_attnres_last.pt`

若任一模型加载失败，直接报错。


In [ ]:
from scripts.evaluate import load_checkpoint

MODEL_ORDER = ["standard", "attnres"]
CHECKPOINTS = {
    "standard": REPO_ROOT / "toygpt2_runs" / "tinystories_dual" / "standard" / "ckpt_standard_last.pt",
    "attnres": REPO_ROOT / "toygpt2_runs" / "tinystories_dual" / "attnres" / "ckpt_attnres_last.pt",
}
device = torch.device("cpu")

models = {}
experiments = {}
checkpoints = {}

for model_name in MODEL_ORDER:
    ckpt_path = CHECKPOINTS[model_name]
    if not ckpt_path.exists():
        raise FileNotFoundError(f"[{model_name}] checkpoint not found: {ckpt_path}")

    model, experiment, checkpoint = load_checkpoint(ckpt_path, device=device)
    model = model.to(device).eval()

    ckpt_model_type = str(checkpoint.get("model_type"))
    if ckpt_model_type != model_name:
        raise RuntimeError(
            f"[{model_name}] checkpoint model_type mismatch: expected={model_name}, got={ckpt_model_type}"
        )

    models[model_name] = model
    experiments[model_name] = experiment
    checkpoints[model_name] = checkpoint

    print(f"[{model_name}] checkpoint: {ckpt_path}")
    print(f"[{model_name}] model_type: {ckpt_model_type}")
    print(f"[{model_name}] step: {checkpoint.get('step')}")
    print("-" * 80)


## 一致性检查（严格）

为保证双模型对照严谨，显式检查并要求一致：
- `vocab_size`
- `block_size`
- `n_layer`
- `tokenizer_name`
- `dataset_type`
- `hf_dataset_name`
- `text_field`
- `train_texts` / `val_texts`
- `block_stride`
- `use_token_cache` / `token_cache_dir`

若不一致直接报错，不做 silent 修正。


In [ ]:
CONSISTENCY_FIELDS = [
    "vocab_size",
    "block_size",
    "n_layer",
    "tokenizer_name",
    "dataset_type",
    "hf_dataset_name",
    "text_field",
    "train_texts",
    "val_texts",
    "block_stride",
    "use_token_cache",
    "token_cache_dir",
]

model_summaries = {}
for model_name in MODEL_ORDER:
    exp = experiments[model_name]
    summary = {
        "model_type": exp.model.model_type,
        "vocab_size": int(exp.model.vocab_size),
        "block_size": int(exp.model.block_size),
        "n_layer": int(exp.model.n_layer),
        "tokenizer_name": getattr(exp.data, "tokenizer_name", None),
        "dataset_type": getattr(exp.data, "dataset_type", None),
        "hf_dataset_name": getattr(exp.data, "hf_dataset_name", None),
        "text_field": getattr(exp.data, "text_field", None),
        "train_texts": getattr(exp.data, "train_texts", None),
        "val_texts": getattr(exp.data, "val_texts", None),
        "block_stride": getattr(exp.data, "block_stride", None),
        "use_token_cache": getattr(exp.data, "use_token_cache", None),
        "token_cache_dir": getattr(exp.data, "token_cache_dir", None),
    }
    model_summaries[model_name] = summary
    print(f"[{model_name}] config summary:")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print("-" * 80)


def assert_field_equal(field_name: str) -> None:
    values = {name: model_summaries[name][field_name] for name in MODEL_ORDER}
    uniq = {repr(v) for v in values.values()}
    if len(uniq) != 1:
        raise RuntimeError(
            f"基础一致性检查失败: field={field_name}, values={values}. "
            "请先保证两模型在该条件下可比。"
        )


for field in CONSISTENCY_FIELDS:
    assert_field_equal(field)

shared_vocab_size = int(model_summaries[MODEL_ORDER[0]]["vocab_size"])
shared_block_size = int(model_summaries[MODEL_ORDER[0]]["block_size"])
shared_n_layer = int(model_summaries[MODEL_ORDER[0]]["n_layer"])
print("[consistency] all required fields are aligned across standard/attnres.")


## 准备 TinyStories 样本、next-token target 与 corruption（Case Study）

关键点：
- `clean_tokens` 作为模型输入，形状 `[1, seq_len]`
- `target_position = seq_len - 1`
- `target_token_id` 来自 dataloader 的 `targets` 在同一位置（teacher-forcing next-token）
- corruption 默认作用在 `target_position` 之前


In [ ]:
from data.data import build_dataloaders
from interp.memorization_runner import MemorizationPatchingRunner

runners = {name: MemorizationPatchingRunner(models[name]) for name in MODEL_ORDER}

CASE_BATCH_SIZE = 8
CASE_SEQ_LEN = int(min(32, shared_block_size))
if CASE_SEQ_LEN < 4:
    raise ValueError(f"CASE_SEQ_LEN={CASE_SEQ_LEN} too small for causal patching.")

base_experiment = experiments[MODEL_ORDER[0]]
sample_model_config = deepcopy(base_experiment.model)
sample_model_config.block_size = shared_block_size

tiny_cfg = deepcopy(base_experiment.data)
tiny_cfg.dataset_type = "tinystories"
tiny_cfg.block_stride = max(1, sample_model_config.block_size)
if tiny_cfg.train_texts is None:
    tiny_cfg.train_texts = 1024
if tiny_cfg.val_texts is None:
    tiny_cfg.val_texts = 256

train_loader, _ = build_dataloaders(
    model_config=sample_model_config,
    data_config=tiny_cfg,
    batch_size=CASE_BATCH_SIZE,
    num_workers=0,
    seed=SEED,
    distributed=False,
    verbose=True,
)


def _assert_token_range(tokens: torch.Tensor, vocab_size: int, name: str) -> None:
    tmin = int(tokens.min().item())
    tmax = int(tokens.max().item())
    if tmin < 0 or tmax >= vocab_size:
        raise RuntimeError(
            f"{name} token id out of range: min={tmin}, max={tmax}, vocab_size={vocab_size}"
        )



def collect_n_samples_from_loader(loader, n: int, seq_len: int):
    input_parts = []
    target_parts = []
    total = 0
    for batch_inputs, batch_targets in loader:
        input_parts.append(batch_inputs[:, :seq_len])
        target_parts.append(batch_targets[:, :seq_len])
        total += int(batch_inputs.size(0))
        if total >= n:
            break

    if not input_parts:
        raise RuntimeError("loader produced zero batches.")

    inputs = torch.cat(input_parts, dim=0)[:n].to(device=device, dtype=torch.long)
    targets = torch.cat(target_parts, dim=0)[:n].to(device=device, dtype=torch.long)
    if inputs.size(0) < n:
        raise RuntimeError(f"requested n={n}, but only got {inputs.size(0)} samples.")
    return inputs, targets



def pick_corruption_token(
    candidate_batch: torch.Tensor,
    sample_index: int,
    corrupt_position: int,
    vocab_size: int,
    rng: np.random.Generator,
) -> tuple[int, str]:
    original = int(candidate_batch[sample_index, corrupt_position].item())
    batch_size, seq_len = candidate_batch.shape

    candidates = []

    # Priority 1: other samples at same position (if available)
    for bi in range(batch_size):
        if bi == sample_index:
            continue
        token_id = int(candidate_batch[bi, corrupt_position].item())
        if token_id != original:
            candidates.append(token_id)

    # Priority 2: same sample at other positions
    for pos in range(seq_len):
        if pos == corrupt_position:
            continue
        token_id = int(candidate_batch[sample_index, pos].item())
        if token_id != original:
            candidates.append(token_id)

    if candidates:
        chosen = int(candidates[int(rng.integers(0, len(candidates)))])
        return chosen, "from_batch_or_context"

    # Fallback: random valid token, but must differ from original
    for _ in range(256):
        chosen = int(rng.integers(0, vocab_size))
        if chosen != original:
            return chosen, "random_fallback"
    raise RuntimeError("failed to sample a valid corruption token.")



def build_corrupted_input(
    clean_input: torch.Tensor,
    target_position: int,
    vocab_size: int,
    rng: np.random.Generator,
    candidate_batch: torch.Tensor | None = None,
    candidate_sample_index: int = 0,
) -> tuple[torch.Tensor, dict]:
    if clean_input.ndim != 2 or clean_input.size(0) != 1:
        raise ValueError("clean_input for case study must be shaped [1, seq_len].")

    seq_len = int(clean_input.size(1))
    if target_position <= 0 or target_position >= seq_len:
        raise ValueError(f"invalid target_position={target_position} for seq_len={seq_len}")

    corrupt_position = target_position - 1
    if not (0 <= corrupt_position < target_position):
        raise ValueError("corrupt_position must be before target_position.")

    reference = candidate_batch if candidate_batch is not None else clean_input
    if reference.ndim != 2:
        raise ValueError("candidate_batch must be shaped [batch, seq_len].")
    if not (0 <= candidate_sample_index < reference.size(0)):
        raise ValueError(
            f"candidate_sample_index out of range: idx={candidate_sample_index}, batch={reference.size(0)}"
        )

    new_token, source = pick_corruption_token(
        candidate_batch=reference,
        sample_index=candidate_sample_index,
        corrupt_position=corrupt_position,
        vocab_size=vocab_size,
        rng=rng,
    )

    original_token = int(clean_input[0, corrupt_position].item())
    if new_token == original_token:
        raise RuntimeError("corruption token equals original token, which is invalid.")
    if not (0 <= new_token < vocab_size):
        raise RuntimeError(f"corruption token out of range: token={new_token}, vocab_size={vocab_size}")

    corrupted = clean_input.clone()
    corrupted[0, corrupt_position] = new_token
    return corrupted, {
        "corrupt_position": int(corrupt_position),
        "original_token_id": int(original_token),
        "corrupted_token_id": int(new_token),
        "corruption_source": source,
    }


# ---- Prepare case-study sample ----
case_inputs_batch, case_targets_batch = collect_n_samples_from_loader(
    train_loader,
    n=CASE_BATCH_SIZE,
    seq_len=CASE_SEQ_LEN,
)
_assert_token_range(case_inputs_batch, shared_vocab_size, "case_inputs_batch")
_assert_token_range(case_targets_batch, shared_vocab_size, "case_targets_batch")

CASE_SAMPLE_INDEX = 0
clean_tokens = case_inputs_batch[CASE_SAMPLE_INDEX : CASE_SAMPLE_INDEX + 1].clone()
clean_targets = case_targets_batch[CASE_SAMPLE_INDEX : CASE_SAMPLE_INDEX + 1].clone()

# Teacher-forcing next-token target
target_position = int(clean_tokens.size(1) - 1)
target_token_id = int(clean_targets[0, target_position].item())

case_rng = np.random.default_rng(SEED + 11)
corrupted_tokens, case_corruption_info = build_corrupted_input(
    clean_input=clean_tokens,
    target_position=target_position,
    vocab_size=shared_vocab_size,
    rng=case_rng,
    candidate_batch=case_inputs_batch,
    candidate_sample_index=CASE_SAMPLE_INDEX,
)

sample_info = {
    "source": "tinystories_train_loader",
    "case_seq_len": int(CASE_SEQ_LEN),
    "shared_vocab_size": int(shared_vocab_size),
    "shared_block_size": int(shared_block_size),
    "target_position": int(target_position),
    "target_token_id_from_targets": int(target_token_id),
    "corruption": case_corruption_info,
}

print("sample_info:")
print(json.dumps(sample_info, ensure_ascii=False, indent=2))
print("clean_tokens:", clean_tokens.tolist())
print("clean_targets(next-token):", clean_targets.tolist())
print("corrupted_tokens:", corrupted_tokens.tolist())


## Case Study：Clean forward、可观测位点与稳定排序的 site sweep

这里先做单样本 case study，并构造稳定、可复现的 sweep site 顺序：
- 先按 block 编号
- 每个 block 内固定顺序：`attn_out -> mlp_out -> output`


In [ ]:
from interp.analysis_adapter import AnalysisAdapter

clean_outputs_by_model = {}
cache_by_model = {}
trace_by_model = {}
candidate_sites_by_model = {}

with torch.no_grad():
    for model_name in MODEL_ORDER:
        outputs = models[model_name](
            clean_tokens,
            return_intermediates=True,
            return_attn=True,
            return_cache=True,
        )
        clean_outputs_by_model[model_name] = outputs

        cache_obj = outputs["cache"]
        cache_dict = cache_obj.data
        cache_by_model[model_name] = cache_dict
        cache_keys = sorted(cache_dict.keys())

        candidate_sites = [
            key for key in cache_keys if key.endswith(("attn_out", "mlp_out", "output"))
        ]
        candidate_sites_by_model[model_name] = candidate_sites

        trace = AnalysisAdapter.from_model_outputs(outputs)
        trace_by_model[model_name] = trace

        print(f"[{model_name}] cache key count: {len(cache_keys)}")
        print(f"[{model_name}] candidate patch sites: {len(candidate_sites)}")
        print("-" * 80)

common_patch_sites = None
for model_name in MODEL_ORDER:
    sites = set(candidate_sites_by_model[model_name])
    common_patch_sites = sites if common_patch_sites is None else (common_patch_sites & sites)
common_patch_sites = sorted(common_patch_sites) if common_patch_sites is not None else []

if not common_patch_sites:
    raise RuntimeError("No common patch sites found across two models.")


def build_ordered_sweep_sites(common_sites: list[str], n_layers: int) -> list[str]:
    ordered = []
    common_set = set(common_sites)
    for layer_idx in range(n_layers):
        for suffix in ("attn_out", "mlp_out", "output"):
            site = f"blocks.{layer_idx}.{suffix}"
            if site in common_set:
                ordered.append(site)
    return ordered


sweep_sites = build_ordered_sweep_sites(common_patch_sites, shared_n_layer)
if not sweep_sites:
    raise RuntimeError(
        "Ordered sweep_sites is empty after filtering common sites. "
        "Please inspect cache keys and layer settings."
    )

preferred_patch_site = "blocks.0.attn_out"
patch_site = preferred_patch_site if preferred_patch_site in sweep_sites else sweep_sites[0]

print("[site selection] total common sites:", len(common_patch_sites))
print("[site selection] ordered sweep_sites:")
for s in sweep_sites:
    print("  ", s)
print("[site selection] selected patch_site:", patch_site)


## Case Study：单样本 clean/corrupted/patched/ablated

这部分是 **单样本个案分析**。所有分数都指向：
**next-token target logprob**。


In [ ]:
from interp.ablation import zero_ablation_override


def next_token_target_logprob(outputs, pos: int, token_id: int) -> float:
    logits = outputs["logits"]
    return float(torch.log_softmax(logits, dim=-1)[0, pos, token_id].item())


results_by_model = {}
corrupted_outputs_by_model = {}
ablated_outputs_by_model = {}
summary_by_model = {}

for model_name in MODEL_ORDER:
    cache = cache_by_model[model_name]
    if patch_site not in cache:
        raise KeyError(f"[{model_name}] patch_site not found in clean cache: {patch_site}")

    runner = runners[model_name]
    result_patch = runner.run(
        clean_input=clean_tokens,
        corrupted_input=corrupted_tokens,
        patch_site=patch_site,
        target_position=target_position,
        target_token_id=target_token_id,
        metric="logprob",
    )

    with torch.no_grad():
        corrupted_outputs = models[model_name](
            corrupted_tokens,
            return_intermediates=True,
            return_cache=True,
        )
        ablated_outputs = models[model_name](
            corrupted_tokens,
            return_intermediates=True,
            return_cache=True,
            activation_overrides=zero_ablation_override(patch_site),
        )

    clean_lp = float(result_patch.baseline_clean_score.item())
    corr_lp = float(result_patch.baseline_corrupted_score.item())
    patch_lp = float(result_patch.patched_score.item())
    ablate_lp = next_token_target_logprob(ablated_outputs, target_position, target_token_id)

    patch_effect = float(result_patch.effect_size.item())

    results_by_model[model_name] = result_patch
    corrupted_outputs_by_model[model_name] = corrupted_outputs
    ablated_outputs_by_model[model_name] = ablated_outputs
    summary_by_model[model_name] = {
        "clean_next_token_logprob": clean_lp,
        "corrupted_next_token_logprob": corr_lp,
        "patched_next_token_logprob": patch_lp,
        "ablated_next_token_logprob": ablate_lp,
        "patch_effect": patch_effect,
    }

    print(f"[{model_name}] patch_site: {patch_site}")
    print(f"[{model_name}] clean next-token target logprob:      {clean_lp:.6f}")
    print(f"[{model_name}] corrupted next-token target logprob:  {corr_lp:.6f}")
    print(f"[{model_name}] patched next-token target logprob:    {patch_lp:.6f}")
    print(f"[{model_name}] ablated next-token target logprob:    {ablate_lp:.6f}")
    print(f"[{model_name}] patch effect (patched - corrupted):   {patch_effect:+.6f}")
    print("-" * 80)

save_json_artifact({'patch_site': patch_site, 'summary_by_model': summary_by_model}, 'case_patch_summary.json')


## Case Study：目标 logprob 与 top-k

继续只看单样本，检查 clean/corrupted/patched/ablated 的 next-token 目标分数与 top-k 分布变化。


In [ ]:
def topk_tokens(outputs, pos: int, k: int = 5):
    logits = outputs["logits"][0, pos]
    probs = torch.softmax(logits, dim=-1)
    vals, idx = torch.topk(probs, k=k)
    return [(int(i), float(v)) for i, v in zip(idx, vals)]


case_rows = []

for model_name in MODEL_ORDER:
    with torch.no_grad():
        clean_outputs_eval = models[model_name](clean_tokens)
        corr_outputs_eval = models[model_name](corrupted_tokens)
        patched_outputs_eval = models[model_name](
            corrupted_tokens,
            activation_overrides={patch_site: clean_outputs_by_model[model_name]["cache"][patch_site].clone()},
        )

    s = summary_by_model[model_name]
    case_rows.extend([
        {
            "model": model_name,
            "condition": "clean",
            "next_token_target_logprob": float(s["clean_next_token_logprob"]),
            "top5": topk_tokens(clean_outputs_eval, target_position),
        },
        {
            "model": model_name,
            "condition": "corrupted",
            "next_token_target_logprob": float(s["corrupted_next_token_logprob"]),
            "top5": topk_tokens(corr_outputs_eval, target_position),
        },
        {
            "model": model_name,
            "condition": "patched_from_clean",
            "next_token_target_logprob": float(s["patched_next_token_logprob"]),
            "top5": topk_tokens(patched_outputs_eval, target_position),
        },
        {
            "model": model_name,
            "condition": "zero_ablated",
            "next_token_target_logprob": float(s["ablated_next_token_logprob"]),
            "top5": topk_tokens(ablated_outputs_by_model[model_name], target_position),
        },
    ])

print(json.dumps(case_rows, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
conditions = [
    ("clean_next_token_logprob", "clean"),
    ("corrupted_next_token_logprob", "corrupted"),
    ("patched_next_token_logprob", "patched"),
    ("ablated_next_token_logprob", "ablated"),
]
labels = []
vals = []
for model_name in MODEL_ORDER:
    for key, short_name in conditions:
        labels.append(f"{model_name}\n{short_name}")
        vals.append(float(summary_by_model[model_name][key]))

ax.bar(range(len(vals)), vals)
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labels, rotation=20)
ax.set_title(
    "Case study next-token target logprob\n"
    f"target_position={target_position}, target_token_id={target_token_id}"
)
ax.set_ylabel("next-token target logprob")
ax.axhline(0.0, color="black", linewidth=1)
plt.tight_layout()
save_figure_artifact('case_next_token_logprob.png', fig)
plt.show()

save_json_artifact({'patch_site': patch_site, 'case_rows': case_rows, 'summary_by_model': summary_by_model}, 'case_study_outputs.json')


## Case Study：按稳定顺序做 site sweep

对同一单样本，在 `sweep_sites` 上分别计算两模型的 patch effect：
`patched next-token target logprob - corrupted next-token target logprob`。


In [ ]:
site_effects_by_model = {}
for model_name in MODEL_ORDER:
    sweep = runners[model_name].run_site_sweep(
        clean_input=clean_tokens,
        corrupted_input=corrupted_tokens,
        patch_sites=sweep_sites,
        target_position=target_position,
        target_token_id=target_token_id,
        metric="logprob",
    )
    site_effects_by_model[model_name] = [float(v.item()) for v in sweep.effect_by_site[:, 0]]

fig, ax = plt.subplots(1, 1, figsize=(max(12, 1.3 * len(sweep_sites)), 4))
base_x = np.arange(len(sweep_sites))
bar_width = 0.8 / max(1, len(MODEL_ORDER))
for mi, model_name in enumerate(MODEL_ORDER):
    offsets = base_x - 0.4 + bar_width / 2 + mi * bar_width
    ax.bar(offsets, site_effects_by_model[model_name], width=bar_width, label=model_name)

ax.set_xticks(base_x)
ax.set_xticklabels(sweep_sites, rotation=30, ha="right")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("patch effect")
ax.set_title(
    "Case study patch effect by site\n"
    "(patched next-token logprob - corrupted next-token logprob)"
)
ax.legend()
plt.tight_layout()
save_figure_artifact('site_effects_by_model.png', fig)
plt.show()

print("case-study site_effects_by_model:")
for model_name in MODEL_ORDER:
    print(f"[{model_name}]")
    for site, effect in zip(sweep_sites, site_effects_by_model[model_name]):
        print(f"  {site:36s} {effect:+.6f}")

save_json_artifact({'sweep_sites': sweep_sites, 'site_effects_by_model': site_effects_by_model}, 'site_effects_by_model.json')


## Aggregate：多样本 site sweep 统计（更稳健）

在同一协议下，对前 `N` 个样本重复：
`clean -> corrupted -> patched`，并统计每个模型、每个 site 的：
- `mean_effect`
- `std_effect`
- `positive_rate`（effect > 0）
- `count`

注意：这部分比单样本 case study 更接近整体趋势，但仍是小规模分析。


In [ ]:
AGG_NUM_SAMPLES = 16  # 可改成 32

aggregate_inputs, aggregate_targets = collect_n_samples_from_loader(
    train_loader,
    n=AGG_NUM_SAMPLES,
    seq_len=CASE_SEQ_LEN,
)
_assert_token_range(aggregate_inputs, shared_vocab_size, "aggregate_inputs")
_assert_token_range(aggregate_targets, shared_vocab_size, "aggregate_targets")

aggregate_records = []

for sample_idx in range(AGG_NUM_SAMPLES):
    clean_i = aggregate_inputs[sample_idx : sample_idx + 1].clone()
    target_i = aggregate_targets[sample_idx : sample_idx + 1].clone()

    target_token_i = int(target_i[0, target_position].item())
    sample_rng = np.random.default_rng(SEED + 1000 + sample_idx)
    corrupted_i, corr_meta_i = build_corrupted_input(
        clean_input=clean_i,
        target_position=target_position,
        vocab_size=shared_vocab_size,
        rng=sample_rng,
        candidate_batch=aggregate_inputs,
        candidate_sample_index=sample_idx,
    )

    for model_name in MODEL_ORDER:
        sweep = runners[model_name].run_site_sweep(
            clean_input=clean_i,
            corrupted_input=corrupted_i,
            patch_sites=sweep_sites,
            target_position=target_position,
            target_token_id=target_token_i,
            metric="logprob",
        )
        effects = [float(v.item()) for v in sweep.effect_by_site[:, 0]]

        for site, effect in zip(sweep_sites, effects):
            aggregate_records.append(
                {
                    "sample_idx": int(sample_idx),
                    "model": model_name,
                    "site": site,
                    "effect": float(effect),
                    "target_token_id": int(target_token_i),
                    "corrupt_position": int(corr_meta_i["corrupt_position"]),
                }
            )

aggregate_effects_df = pd.DataFrame(aggregate_records)
if aggregate_effects_df.empty:
    raise RuntimeError("aggregate_effects_df is empty; cannot summarize.")

summary_basic = (
    aggregate_effects_df.groupby(["model", "site"], as_index=False)["effect"]
    .agg(mean_effect="mean", std_effect="std", count="count")
)
summary_pos = (
    aggregate_effects_df.assign(is_positive=aggregate_effects_df["effect"] > 0)
    .groupby(["model", "site"], as_index=False)["is_positive"]
    .mean()
    .rename(columns={"is_positive": "positive_rate"})
)
aggregate_summary_df = summary_basic.merge(summary_pos, on=["model", "site"], how="left")
aggregate_summary_df["std_effect"] = aggregate_summary_df["std_effect"].fillna(0.0)

site_rank = {site: i for i, site in enumerate(sweep_sites)}
model_rank = {model: i for i, model in enumerate(MODEL_ORDER)}
aggregate_summary_df["_site_rank"] = aggregate_summary_df["site"].map(site_rank)
aggregate_summary_df["_model_rank"] = aggregate_summary_df["model"].map(model_rank)
aggregate_summary_df = (
    aggregate_summary_df.sort_values(["_site_rank", "_model_rank"])
    .drop(columns=["_site_rank", "_model_rank"])
    .reset_index(drop=True)
)

print(f"aggregate sample count: {AGG_NUM_SAMPLES}")
print("aggregate_summary_df:")
print(aggregate_summary_df.to_string(index=False))

fig, ax = plt.subplots(1, 1, figsize=(max(12, 1.3 * len(sweep_sites)), 4))
x = np.arange(len(sweep_sites))
w = 0.8 / max(1, len(MODEL_ORDER))

for mi, model_name in enumerate(MODEL_ORDER):
    sub = aggregate_summary_df[aggregate_summary_df["model"] == model_name]
    mean_map = {row["site"]: float(row["mean_effect"]) for _, row in sub.iterrows()}
    std_map = {row["site"]: float(row["std_effect"]) for _, row in sub.iterrows()}

    y = [mean_map[s] for s in sweep_sites]
    yerr = [std_map[s] for s in sweep_sites]
    offsets = x - 0.4 + w / 2 + mi * w
    ax.bar(offsets, y, width=w, label=model_name, yerr=yerr, capsize=2)

ax.set_xticks(x)
ax.set_xticklabels(sweep_sites, rotation=30, ha="right")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("mean patch effect")
ax.set_title(
    "Aggregate mean patch effect by site (error bar = std)\n"
    "effect = patched next-token logprob - corrupted next-token logprob"
)
ax.legend()
plt.tight_layout()
save_figure_artifact('aggregate_site_effects.png', fig)
plt.show()

save_dataframe_artifact(aggregate_summary_df, 'aggregate_summary.csv')
save_json_artifact(aggregate_records, 'aggregate_records.json')
save_json_artifact(
    {
        'agg_num_samples': AGG_NUM_SAMPLES,
        'sweep_sites': sweep_sites,
        'aggregate_summary': aggregate_summary_df.to_dict(orient='records'),
    },
    'aggregate_summary.json',
)


## 结果解释（避免过度结论）

这份 notebook 完成的是双模型 block/site 级 causal patching 分析：
- **Case Study（单样本）**：用于展示具体样本下的可解释现象
- **Aggregate（多样本）**：提供更稳健的趋势性统计（mean/std/positive-rate/count）

因此本 notebook 可以支持的结论是：
- 在统一输入协议与统一 site 顺序下，可对 `standard` 与 `attnres` 的 patch effect 做可复现比较。

本 notebook 不直接给出“最终 memorization 强弱定论”；
若要形成更强结论，还需要更大样本规模与更完整实验设计。
